# Validação e benchmark ETL vs ELT — CPGF (Colab, com upload dos CSVs)

Este notebook **executa os dois pipelines** do projeto (`ETL.ipynb` e `ELT.ipynb`), **mede** o tempo e a memória de cada um e depois **valida** os bancos gerados.

**Como usar:** na primeira execução deixe `MODO_AMOSTRA = True` (leva cerca de 1 minuto). Depois, para a base completa, mude para `False` e **envie os 3 CSVs do seu computador** pelo botão de upload que aparece na seção 1 (o envio **não entra na medição de tempo**). Use **Ambiente de execução > Executar tudo**.

## 1. Preparando o ambiente

No Colab, a célula abaixo clona o repositório em `/content/projeto-integracao-dw`. Fora do Colab, o notebook procura a raiz do repositório a partir da pasta atual.

In [ ]:
import json
import re
import statistics
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

try:
    import psutil
except ImportError:  # fora do Colab pode não estar instalado
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil'], check=True)
    import psutil

import csv
import sqlite3

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

EM_COLAB = 'google.colab' in sys.modules
URL_REPOSITORIO = 'https://github.com/fagpaf/projeto-integracao-dw'
RAMO_REPOSITORIO = 'main'
PASTA_COLAB = Path('/content/projeto-integracao-dw')


def git(*argumentos, cwd=None):
    subprocess.run(['git', *argumentos], cwd=cwd, check=True)


if EM_COLAB:
    if (PASTA_COLAB / '.git').is_dir():
        git('fetch', 'origin', RAMO_REPOSITORIO, cwd=PASTA_COLAB)
        git('checkout', '-B', RAMO_REPOSITORIO, f'origin/{RAMO_REPOSITORIO}', cwd=PASTA_COLAB)
    else:
        git('clone', '--branch', RAMO_REPOSITORIO, URL_REPOSITORIO, str(PASTA_COLAB))
    print('Repositório pronto em', PASTA_COLAB, '- ramo', RAMO_REPOSITORIO)
else:
    print('Fora do Colab: usando o repositório local.')

### Configuração

- `MODO_AMOSTRA`: `True` usa as amostras versionadas (300 linhas); `False` usa a base completa (cerca de 438 mil linhas).
- `REPETICOES`: quantas vezes cada pipeline é executado. Com mais de uma repetição, o relatório mostra mínimo e mediana, o que reduz o efeito de oscilações do ambiente (cache de disco, outros processos). Na base completa, cada repetição pode levar vários minutos.
- `TIMEOUT_S`: tempo máximo de cada célula executada pelo nbconvert (não da execução inteira).

In [2]:
MODO_AMOSTRA = False
REPETICOES = 3
ANOS = (2023, 2024, 2025)
TIMEOUT_S = 3600
RAIZ_CONFIGURADA = PASTA_COLAB if EM_COLAB else None


def localizar_raiz(configurada=None):
    candidatos = [Path(configurada)] if configurada is not None else [Path.cwd(), *Path.cwd().parents]
    for pasta in candidatos:
        if (pasta / 'scripts/extract_cpgf.py').is_file():
            return pasta.resolve()
    raise FileNotFoundError('Configure RAIZ_CONFIGURADA com a pasta do repositório.')


RAIZ = localizar_raiz(RAIZ_CONFIGURADA)
SUFIXO = '_amostra' if MODO_AMOSTRA else ''
DB_ETL = RAIZ / 'data/database' / f'cpgf_dw{SUFIXO}.sqlite'
DB_ELT = RAIZ / 'data/database' / f'cpgf_elt{SUFIXO}.sqlite'
ARQUIVOS_CSV = [
    RAIZ / 'data/raw' / (f'amostras/{ano}_amostra.csv' if MODO_AMOSTRA else f'{ano}_CPGF.csv')
    for ano in ANOS
]
print('Modo:', 'amostra' if MODO_AMOSTRA else 'completo', '| repetições:', REPETICOES)
print('Raiz:', RAIZ)

Modo: completo | repetições: 3
Raiz: /home/fagpaf/Projetos/projeto-integracao-dw


### Enviando os CSVs da base completa (`MODO_AMOSTRA = False`)

Os CSVs completos **não ficam no repositório** (o tamanho estimado é de uns 140 MB). Com `MODO_AMOSTRA = True` você pode ignorar esta parte. Com `False`, os arquivos são enviados do **seu computador** para o Colab:

1. **Consiga os 3 arquivos.** No seu computador, na raiz do repositório, rode `python scripts/extract_cpgf.py`. Ele gera `2023_CPGF.csv`, `2024_CPGF.csv` e `2025_CPGF.csv` na pasta `data/raw/`.
2. **Execute a célula "Preparando as entradas"** (a próxima). Ela vai parar e mostrar o botão **Escolher arquivos**.
3. **Selecione os 3 CSVs de uma vez** (Ctrl ou Cmd + clique) e **espere o envio terminar**. A célula continua "rodando" até acabar, e isso pode levar alguns minutos.
4. Ao terminar, a célula mostra o tamanho de cada arquivo. Confira se os três aparecem com tamanho maior que zero.

Observações:

- Os nomes precisam ser **exatamente** `2023_CPGF.csv`, `2024_CPGF.csv` e `2025_CPGF.csv`. Arquivos com outros nomes são ignorados, com um aviso. Se o Colab acrescentar ` (1)` ao nome (quando você envia o mesmo arquivo duas vezes), o notebook corrige sozinho.
- Se você executar a célula de novo **na mesma sessão**, ela não pede o envio outra vez, porque os arquivos já estão lá.
- Quando a sessão do Colab termina ou desconecta, os arquivos somem, e será preciso enviar de novo.
- Este botão de envio só existe no Colab.


### Preparando as entradas (fora da medição)

Na base completa, os arquivos enviados por você precisam estar na pasta do projeto antes do benchmark. O envio é feito **antes** de medir. Se o envio ficasse dentro do primeiro pipeline executado, o tempo do upload seria atribuído ao ETL e distorceria a comparação.

Também garantimos que existe um kernel `python3` para o `nbconvert` executar os notebooks.

In [3]:
import shutil


def garantir_kernel():
    saida = subprocess.run([sys.executable, '-m', 'jupyter', 'kernelspec', 'list', '--json'],
                           capture_output=True, text=True)
    if saida.returncode == 0 and 'python3' in json.loads(saida.stdout).get('kernelspecs', {}):
        return
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipykernel', 'nbconvert'], check=True)
    subprocess.run([sys.executable, '-m', 'ipykernel', 'install', '--user', '--name', 'python3'], check=True)


garantir_kernel()

PASTA_RAW = ARQUIVOS_CSV[0].parent


def receber_por_upload():
    from google.colab import files
    PASTA_RAW.mkdir(parents=True, exist_ok=True)
    esperados = {c.name for c in ARQUIVOS_CSV}
    print('Selecione os 3 arquivos:', sorted(esperados))
    enviados = files.upload()
    if not enviados:
        raise RuntimeError('Nenhum arquivo foi enviado.')
    for nome in enviados:
        # O Colab acrescenta " (1)" ao nome quando já existe um arquivo igual na pasta de trabalho.
        limpo = re.sub(r' \(\d+\)(?=\.[^.]+$)', '', nome)
        if limpo in esperados or limpo == 'metadados.json':
            shutil.move(nome, PASTA_RAW / limpo)
        else:
            Path(nome).unlink(missing_ok=True)
            print('Ignorado (nome inesperado):', nome)


def conferir_csv(caminho):
    """Confere se o arquivo não está vazio e se o cabeçalho é o do CPGF (evita usar um arquivo errado)."""
    if caminho.stat().st_size == 0:
        raise ValueError(f'{caminho.name} está vazio. O envio pode ter sido interrompido; envie de novo.')
    try:
        with caminho.open(encoding='utf-8-sig') as arquivo:
            cabecalho = arquivo.readline()
    except UnicodeDecodeError:
        raise ValueError(f'{caminho.name} não está em UTF-8. Use o arquivo gerado por scripts/extract_cpgf.py.')
    if 'VALOR TRANSAÇÃO' not in cabecalho:
        raise ValueError(f'{caminho.name} não parece o CSV anual do CPGF (cabeçalho inesperado): {cabecalho[:80]!r}')


faltando = [c for c in ARQUIVOS_CSV if not c.is_file()]
if faltando and not MODO_AMOSTRA:
    if not EM_COLAB:
        raise RuntimeError('O botão de upload só funciona no Colab. Fora dele, copie os 3 CSVs para data/raw.')
    receber_por_upload()
    ainda_faltando = [c.name for c in ARQUIVOS_CSV if not c.is_file()]
    if ainda_faltando:
        raise FileNotFoundError(f'Ainda faltam: {ainda_faltando}. Execute a célula de novo e selecione os 3 arquivos.')

for csv_anual in ARQUIVOS_CSV:
    conferir_csv(csv_anual)
    print(f'{csv_anual.name}: {csv_anual.stat().st_size / 1e6:.1f} MB')
print('CSVs disponíveis:', [c.name for c in ARQUIVOS_CSV])

2023_CPGF.csv: 37.8 MB
2024_CPGF.csv: 39.9 MB
2025_CPGF.csv: 45.0 MB
CSVs disponíveis: ['2023_CPGF.csv', '2024_CPGF.csv', '2025_CPGF.csv']


## 2. Benchmark: executando e medindo os pipelines

Para cada pipeline, o notebook original é **copiado** para um arquivo temporário com o `MODO_AMOSTRA` ajustado, e executado com `jupyter nbconvert --execute`. Os notebooks originais não são alterados. Nessas cópias, a detecção do Colab (`EM_COLAB`) é desligada: o repositório já foi clonado na seção 1, e o `ELT.ipynb` não deve clonar nem trocar de ramo no meio da medição.

O que é medido:

- **Tempo total (parede):** do início ao fim da execução, incluindo a partida do kernel Python (um custo parecido para os dois).
- **Tempo por célula:** o `nbconvert` registra o horário de início e fim de cada célula. Agrupamos por seção do notebook para ver **em que etapa** o tempo é gasto.
- **Pico de memória:** a cada 50 ms somamos a memória residente (RSS) do kernel e de seus subprocessos, e guardamos o maior valor. É uma amostragem: picos mais curtos que 50 ms podem passar.

A ordem alterna entre ETL/ELT e ELT/ETL a cada repetição para reduzir o efeito da ordem sobre a medição. Os dois recriam seus bancos do zero, então repetir não acumula dados.

In [ ]:
def preparar_copia(nome):
    origem = RAIZ / 'notebooks' / f'{nome}.ipynb'
    destino = RAIZ / 'notebooks' / f'_bench_{nome}.ipynb'
    texto, trocas = re.subn(r'MODO_AMOSTRA = (?:True|False)', f'MODO_AMOSTRA = {MODO_AMOSTRA}',
                            origem.read_text(encoding='utf-8'))
    if trocas == 0:
        raise ValueError(f'{origem.name}: não encontrei a linha MODO_AMOSTRA.')
    # No Colab, o kernel que o nbconvert abre também enxerga o módulo google.colab. Sem esta troca,
    # ETL.ipynb e ELT.ipynb achariam que estão sozinhos no Colab e tentariam clonar/atualizar o
    # repositório por conta própria, no meio da medição. Forçamos EM_COLAB = False: o repositório
    # já foi clonado acima.
    texto = re.sub(r"EM_COLAB = 'google\.colab' in sys\.modules", 'EM_COLAB = False', texto)
    destino.write_text(texto, encoding='utf-8')
    return destino


def executar(nome):
    copia = preparar_copia(nome)
    saida = copia.with_name(f'_bench_{nome}_saida.ipynb')
    log = copia.with_name(f'_bench_{nome}.log')
    comando = [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', copia.name,
               '--output', saida.name, f'--ExecutePreprocessor.timeout={TIMEOUT_S}']
    pico = 0
    inicio = time.perf_counter()
    with log.open('w') as arquivo_log:
        processo = subprocess.Popen(comando, cwd=RAIZ / 'notebooks', stdout=arquivo_log, stderr=subprocess.STDOUT)
        raiz_proc = psutil.Process(processo.pid)
        while processo.poll() is None:
            try:
                # Só os descendentes (kernel e subprocessos): exclui o processo do próprio nbconvert.
                memoria = sum(p.memory_info().rss for p in raiz_proc.children(recursive=True))
                pico = max(pico, memoria)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
            time.sleep(0.05)
    duracao = time.perf_counter() - inicio
    if processo.returncode != 0:
        raise RuntimeError(f'{nome} falhou. Últimas linhas do log:\n' + log.read_text()[-1500:])
    return {'pipeline': nome, 'tempo_total_s': duracao, 'pico_memoria_mb': pico / 1e6, 'saida': saida}


def tempos_por_celula(saida):
    """Duração de cada célula de código, agrupada pelo último título (## ...) que a precede."""
    notebook = json.loads(saida.read_text(encoding='utf-8'))
    secao, linhas = 'Início', []
    converter = lambda t: datetime.fromisoformat(t.replace('Z', '+00:00'))
    for celula in notebook['cells']:
        fonte = ''.join(celula['source'])
        if celula['cell_type'] == 'markdown':
            titulos = [l.lstrip('# ').strip() for l in fonte.splitlines() if l.startswith('#')]
            if titulos:
                secao = titulos[0]
            continue
        execucao = celula.get('metadata', {}).get('execution', {})
        if 'iopub.execute_input' in execucao and 'shell.execute_reply' in execucao:
            duracao = (converter(execucao['shell.execute_reply']) - converter(execucao['iopub.execute_input'])).total_seconds()
            primeira = next((l.strip() for l in fonte.splitlines() if l.strip() and not l.strip().startswith('#')), '')
            linhas.append({'secao': secao, 'celula': primeira[:60], 'segundos': duracao})
    return pd.DataFrame(linhas)


execucoes, celulas = [], {}
for repeticao in range(1, REPETICOES + 1):
    for nome in (('ETL', 'ELT') if repeticao % 2 else ('ELT', 'ETL')):
        resultado = executar(nome)
        resultado['repeticao'] = repeticao
        execucoes.append(resultado)
        celulas[nome] = tempos_por_celula(resultado['saida'])  # fica a da última repetição
        print(f'[{repeticao}/{REPETICOES}] {nome}: {resultado["tempo_total_s"]:.1f} s | '
              f'pico de memória {resultado["pico_memoria_mb"]:.0f} MB')

medidas = pd.DataFrame([{k: v for k, v in e.items() if k != 'saida'} for e in execucoes])
resumo_tempo = medidas.groupby('pipeline').agg(
    minimo_s=('tempo_total_s', 'min'), mediana_s=('tempo_total_s', 'median'), maximo_s=('tempo_total_s', 'max'),
    pico_memoria_mb=('pico_memoria_mb', 'max')).round(2)
display(medidas.round(2))
display(resumo_tempo)

### Tempo por etapa

Agrupamos o tempo das células pela seção do notebook (última repetição). As etapas **não são idênticas** entre os pipelines: o ETL transforma em pandas, e o ELT separa "carregar o bruto" de "transformar em SQL". Por isso a comparação mais justa é a do total, e esta tabela serve para ver **onde** cada um gasta tempo. A soma das células é menor que o tempo total, porque o total inclui a partida do kernel.

In [5]:
for nome in ('ETL', 'ELT'):
    df = celulas[nome]
    por_secao = df.groupby('secao', sort=False)['segundos'].sum().round(2).to_frame()
    por_secao['% das células'] = (100 * por_secao['segundos'] / por_secao['segundos'].sum()).round(1)
    print(f'--- {nome}: soma das células = {df["segundos"].sum():.2f} s ---')
    display(por_secao)
    print('Células mais lentas:')
    display(df.nlargest(3, 'segundos').round(2)[['secao', 'celula', 'segundos']])

--- ETL: soma das células = 12.95 s ---


,segundos,% das células
secao,,
1. Preparando o ambiente,0.22,1.7
2. Extraindo os dados,0.00,0.0
Lendo e juntando os três anos,1.41,10.9
Conferindo os metadados da extração,0.13,1.0
3. Conhecendo os dados antes do tratamento,0.40,3.1
Verificando campos vazios e tipos de transação,0.89,6.9
4. Transformando os dados,0.77,5.9
Identificando registros sigilosos,0.25,1.9
Convertendo datas e o mês do extrato,1.02,7.9


Células mais lentas:


,secao,celula,segundos
21,Executando a carga,inicio_carga = time.perf_counter(),5.53
2,Lendo e juntando os três anos,lista_dataframes = [],1.41
9,Convertendo datas e o mês do extrato,"datas = pd.to_datetime(dados['data_original'],...",1.02


--- ELT: soma das células = 16.45 s ---


,segundos,% das células
secao,,
1. Preparando o ambiente,0.01,0.1
2. Definindo a leitura dos dados brutos,0.00,0.0
3. Conferindo os arquivos de entrada,0.00,0.0
4. Carregando a camada Raw,4.88,29.7
5. Conferindo o resultado,0.00,0.0
6. Criando o esquema estrela dentro do banco,0.00,0.0
7. Transformando e carregando as dimensões e a fato,11.54,70.2
8. Conferindo o resultado,0.00,0.0


Células mais lentas:


,secao,celula,segundos
10,7. Transformando e carregando as dimensões e a...,inicio_transformacao = time.perf_counter(),11.53
6,4. Carregando a camada Raw,inicio = time.perf_counter(),4.88
5,4. Carregando a camada Raw,"def validar_tabela(conexao, tabela, caminho, q...",0.01


## 3. Conexão com os bancos e registro das verificações

A partir daqui, os bancos gerados pela **última repetição** são abertos em **modo somente leitura** e anexados a uma conexão em memória com os nomes `etl` e `elt`. Assim, uma consulta pode olhar os dois ao mesmo tempo, e nada é alterado.

Cada verificação é guardada como `OK` ou `FALHA`, e o resumo aparece no fim.

In [ ]:
for banco in (DB_ETL, DB_ELT):
    if not banco.is_file():
        raise FileNotFoundError(f'{banco.name} não foi gerado.')

conexao = sqlite3.connect(':memory:', uri=True)
conexao.execute('ATTACH DATABASE ? AS etl', (f'{DB_ETL.resolve().as_uri()}?mode=ro',))
conexao.execute('ATTACH DATABASE ? AS elt', (f'{DB_ELT.resolve().as_uri()}?mode=ro',))

FATO = 'FATO_TRANSACAO_CPGF'
TABELAS_DIM = ['DIM_TEMPO', 'DIM_MES_EXTRATO', 'DIM_ORGAO', 'DIM_PORTADOR', 'DIM_FAVORECIDO', 'DIM_TIPO_TRANSACAO']
resultados = []


def registrar(bloco, verificacao, ok, detalhe=''):
    resultados.append({'bloco': bloco, 'verificacao': verificacao,
                       'status': 'OK' if ok else 'FALHA', 'detalhe': detalhe})
    print(f'[{"OK" if ok else "FALHA"}] {verificacao}' + (f' — {detalhe}' if detalhe else ''))


def escalar(sql):
    return conexao.execute(sql).fetchone()[0]


def ler_csv(caminho):
    """Leitura independente dos pipelines: só a biblioteca csv, sem pandas e sem SQL de transformação."""
    with Path(caminho).open(encoding='utf-8-sig', newline='') as arquivo:
        for registro in csv.DictReader(arquivo, delimiter=';'):
            yield registro


def para_centavos(texto):
    return int(texto.strip().replace('R$', '').strip().replace('.', '').replace(',', ''))


linhas_csv, centavos_csv = 0, 0
for caminho in ARQUIVOS_CSV:
    for registro in ler_csv(caminho):
        linhas_csv += 1
        centavos_csv += para_centavos(registro['VALOR TRANSAÇÃO'])
print(f'CSVs: {linhas_csv:,} linhas | {centavos_csv:,} centavos')

CSVs: 438,771 linhas | 32,477,451,409 centavos


## 4. Validação da contagem de linhas

Três perguntas:

1. A fato de cada pipeline tem **o mesmo número de linhas dos CSVs**? Nenhuma linha pode ser descartada, nem as repetições exatas.
2. No ELT, as tabelas **raw** (cópia fiel dos CSVs) somam o mesmo total? Isso prova que o Extract & Load não perdeu nada antes da transformação.
3. Cada dimensão tem a **mesma quantidade** nos dois bancos?

In [7]:
fato_etl = escalar(f'SELECT COUNT(*) FROM etl.{FATO}')
fato_elt = escalar(f'SELECT COUNT(*) FROM elt.{FATO}')
raw_elt = sum(escalar(f'SELECT COUNT(*) FROM elt.raw_cpgf_{ano}') for ano in ANOS)
stg_elt = escalar('SELECT COUNT(*) FROM elt.stg_cpgf_transacoes')

registrar('4 contagem', 'fato do ETL = linhas dos CSVs', fato_etl == linhas_csv, f'{fato_etl:,} vs {linhas_csv:,}')
registrar('4 contagem', 'fato do ELT = linhas dos CSVs', fato_elt == linhas_csv, f'{fato_elt:,} vs {linhas_csv:,}')
registrar('4 contagem', 'ELT: camada raw (3 tabelas) = linhas dos CSVs', raw_elt == linhas_csv, f'{raw_elt:,} vs {linhas_csv:,}')
registrar('4 contagem', 'ELT: staging = linhas dos CSVs', stg_elt == linhas_csv, f'{stg_elt:,} vs {linhas_csv:,}')

linhas = [{'tabela': 'CSVs (referência)', 'ETL': linhas_csv, 'ELT': linhas_csv, 'igual': True}]
for tabela in TABELAS_DIM + [FATO]:
    n_etl = escalar(f'SELECT COUNT(*) FROM etl.{tabela}')
    n_elt = escalar(f'SELECT COUNT(*) FROM elt.{tabela}')
    linhas.append({'tabela': tabela, 'ETL': n_etl, 'ELT': n_elt, 'igual': n_etl == n_elt})
    if tabela != FATO:
        registrar('4 contagem', f'{tabela}: mesma quantidade nos dois bancos', n_etl == n_elt, f'ETL={n_etl:,} | ELT={n_elt:,}')
display(pd.DataFrame(linhas))

[OK] fato do ETL = linhas dos CSVs — 438,771 vs 438,771
[OK] fato do ELT = linhas dos CSVs — 438,771 vs 438,771
[OK] ELT: camada raw (3 tabelas) = linhas dos CSVs — 438,771 vs 438,771
[OK] ELT: staging = linhas dos CSVs — 438,771 vs 438,771
[OK] DIM_TEMPO: mesma quantidade nos dois bancos — ETL=1,462 | ELT=1,462
[OK] DIM_ORGAO: mesma quantidade nos dois bancos — ETL=1,374 | ELT=1,374
[OK] DIM_PORTADOR: mesma quantidade nos dois bancos — ETL=9,955 | ELT=9,955
[OK] DIM_FAVORECIDO: mesma quantidade nos dois bancos — ETL=73,665 | ELT=73,665
[OK] DIM_TIPO_TRANSACAO: mesma quantidade nos dois bancos — ETL=8 | ELT=8


,tabela,ETL,ELT,igual
0,CSVs (referência),438771,438771,True
1,DIM_TEMPO,1462,1462,True
2,DIM_ORGAO,1374,1374,True
3,DIM_PORTADOR,9955,9955,True
4,DIM_FAVORECIDO,73665,73665,True
5,DIM_TIPO_TRANSACAO,8,8,True
6,FATO_TRANSACAO_CPGF,438771,438771,True


## 5. Validação da soma financeira (`SUM(valor)`)

Mostramos `SUM(valor_transacao)` como pedido, mas a **comparação é feita em centavos inteiros** (`ROUND(valor * 100)` antes de somar). O motivo é que o SQLite guarda parte dos valores como `REAL` (ponto flutuante), e somar milhares de números assim pode acumular diferenças de centésimos que não são erro de dados. Em centavos inteiros a soma é exata.

As três fontes precisam coincidir: CSVs, ETL e ELT.

In [ ]:
def soma(esquema):
    return conexao.execute(f"""
        SELECT SUM(valor_transacao),
               SUM(CAST(ROUND(valor_transacao * 100) AS INTEGER))
        FROM {esquema}.{FATO}
    """).fetchone()


soma_etl, centavos_etl = soma('etl')
soma_elt, centavos_elt = soma('elt')

registrar('5 soma', 'soma em centavos: ETL = CSVs', centavos_etl == centavos_csv, f'{centavos_etl:,} vs {centavos_csv:,}')
registrar('5 soma', 'soma em centavos: ELT = CSVs', centavos_elt == centavos_csv, f'{centavos_elt:,} vs {centavos_csv:,}')
registrar('5 soma', 'soma em centavos: ETL = ELT', centavos_etl == centavos_elt)

display(pd.DataFrame([
    {'fonte': 'CSVs', 'SUM(valor) em R$ (float)': None, 'total em centavos': centavos_csv, 'total em R$ (exato)': f'{centavos_csv / 100:,.2f}'},
    {'fonte': 'ETL', 'SUM(valor) em R$ (float)': round(soma_etl, 2), 'total em centavos': centavos_etl, 'total em R$ (exato)': f'{centavos_etl / 100:,.2f}'},
    {'fonte': 'ELT', 'SUM(valor) em R$ (float)': round(soma_elt, 2), 'total em centavos': centavos_elt, 'total em R$ (exato)': f'{centavos_elt / 100:,.2f}'},
]))

# Soma por ano do extrato: um ano com erro compensado por outro não passaria despercebido.
por_ano = {}
for esquema in ('etl', 'elt'):
    por_ano[esquema] = dict(conexao.execute(f"""
        SELECT id_mes_extrato / 100, SUM(CAST(ROUND(valor_transacao * 100) AS INTEGER))
        FROM {esquema}.{FATO} GROUP BY 1
    """).fetchall())
csv_por_ano = {}
for caminho in ARQUIVOS_CSV:
    for registro in ler_csv(caminho):
        ano = int(registro['ANO EXTRATO'])
        csv_por_ano[ano] = csv_por_ano.get(ano, 0) + para_centavos(registro['VALOR TRANSAÇÃO'])
tabela_ano = pd.DataFrame({'CSVs': csv_por_ano, 'ETL': por_ano['etl'], 'ELT': por_ano['elt']}).sort_index()
registrar('5 soma', 'soma por ano do extrato: CSVs = ETL = ELT', bool(((tabela_ano['CSVs'] == tabela_ano['ETL']) & (tabela_ano['CSVs'] == tabela_ano['ELT'])).all()))
display(tabela_ano)

[OK] soma em centavos: ETL = CSVs — 32,477,451,409 vs 32,477,451,409
[OK] soma em centavos: ELT = CSVs — 32,477,451,409 vs 32,477,451,409
[OK] soma em centavos: ETL = ELT


,fonte,SUM(valor) em R$ (float),total em centavos,total em R$ (exato)
0,CSVs,NaN,32477451409,"324,774,514.09"
1,ETL,3.247745e+08,32477451409,"324,774,514.09"
2,ELT,3.247745e+08,32477451409,"324,774,514.09"


[OK] soma por ano do extrato: CSVs = ETL = ELT


,CSVs,ETL,ELT
2023,9071721079,9071721079,9071721079
2024,11180259928,11180259928,11180259928
2025,12225470402,12225470402,12225470402


## 6. Validação da integridade referencial

A fato tem 6 chaves estrangeiras, uma para cada dimensão. `id_data_transacao` referencia `DIM_TEMPO`, enquanto `id_mes_extrato` referencia `DIM_MES_EXTRATO`. Fazemos duas verificações **independentes**:

1. `PRAGMA foreign_key_check`: o próprio SQLite varre as FKs declaradas no `schema.sql`.
2. **Contagem de órfãos por chave**, com `LEFT JOIN`: conta as linhas da fato cuja chave não existe na dimensão. Mesmo que alguém esqueça de declarar uma FK no esquema, esta checagem pega.

Também rodamos `PRAGMA integrity_check`, que confere se o arquivo SQLite está íntegro. **Esperado: zero violações em tudo.**

In [ ]:
CHAVES = [
    ('id_data_transacao', 'DIM_TEMPO', 'id_tempo'),
    ('id_mes_extrato', 'DIM_MES_EXTRATO', 'id_mes_extrato'),
    ('id_orgao', 'DIM_ORGAO', 'id_orgao'),
    ('id_portador', 'DIM_PORTADOR', 'id_portador'),
    ('id_favorecido', 'DIM_FAVORECIDO', 'id_favorecido'),
    ('id_tipo_transacao', 'DIM_TIPO_TRANSACAO', 'id_tipo_transacao'),
]

linhas = []
for esquema in ('etl', 'elt'):
    nome = esquema.upper()
    fk = len(conexao.execute(f'PRAGMA {esquema}.foreign_key_check').fetchall())
    integridade = conexao.execute(f'PRAGMA {esquema}.integrity_check').fetchall() == [('ok',)]
    registrar('6 integridade', f'{nome}: PRAGMA foreign_key_check sem violações', fk == 0, f'{fk} violações')
    registrar('6 integridade', f'{nome}: PRAGMA integrity_check ok', integridade)
    for coluna, dimensao, pk in CHAVES:
        orfaos = escalar(f"""
            SELECT COUNT(*) FROM {esquema}.{FATO} f
            LEFT JOIN {esquema}.{dimensao} d ON d.{pk} = f.{coluna}
            WHERE d.{pk} IS NULL
        """)
        linhas.append({'banco': nome, 'chave da fato': coluna, 'dimensão': dimensao, 'órfãos': orfaos})
        registrar('6 integridade', f'{nome}: {coluna} → {dimensao} sem órfãos', orfaos == 0, f'{orfaos} órfãos')
display(pd.DataFrame(linhas).pivot(index='chave da fato', columns='banco', values='órfãos'))

[OK] ETL: PRAGMA foreign_key_check sem violações — 0 violações
[OK] ETL: PRAGMA integrity_check ok
[OK] ETL: id_data_transacao → DIM_TEMPO sem órfãos — 0 órfãos
[OK] ETL: id_mes_extrato → DIM_TEMPO sem órfãos — 0 órfãos
[OK] ETL: id_orgao → DIM_ORGAO sem órfãos — 0 órfãos
[OK] ETL: id_portador → DIM_PORTADOR sem órfãos — 0 órfãos
[OK] ETL: id_favorecido → DIM_FAVORECIDO sem órfãos — 0 órfãos
[OK] ETL: id_tipo_transacao → DIM_TIPO_TRANSACAO sem órfãos — 0 órfãos
[OK] ELT: PRAGMA foreign_key_check sem violações — 0 violações
[OK] ELT: PRAGMA integrity_check ok
[OK] ELT: id_data_transacao → DIM_TEMPO sem órfãos — 0 órfãos
[OK] ELT: id_mes_extrato → DIM_TEMPO sem órfãos — 0 órfãos
[OK] ELT: id_orgao → DIM_ORGAO sem órfãos — 0 órfãos
[OK] ELT: id_portador → DIM_PORTADOR sem órfãos — 0 órfãos
[OK] ELT: id_favorecido → DIM_FAVORECIDO sem órfãos — 0 órfãos
[OK] ELT: id_tipo_transacao → DIM_TIPO_TRANSACAO sem órfãos — 0 órfãos


banco,ELT,ETL
chave da fato,,
id_data_transacao,0,0
id_favorecido,0,0
id_mes_extrato,0,0
id_orgao,0,0
id_portador,0,0
id_tipo_transacao,0,0


### Equivalência do conteúdo e sigilo
Comparamos dimensões e transações pelas informações de negócio, sem exigir SKs iguais. O agrupamento inclui a multiplicidade para detectar perdas ou duplicações compensadas. As verificações de órfãos anteriores complementam os JOINs.


In [ ]:
# As SKs podem diferir: comparamos atributos e chaves de negócio.
def colunas_negocio(esquema, tabela):
    return [r[1] for r in conexao.execute(f'PRAGMA {esquema}.table_info("{tabela}")')
            if not r[5]]  # exclui a chave primária substituta


def diferencas_bidirecionais(esquerda, direita):
    # Cada consulta já agrupa as linhas com COUNT(*), preservando multiplicidades.
    return sum(escalar(f'SELECT COUNT(*) FROM ({a} EXCEPT {b})')
               for a, b in ((esquerda, direita), (direita, esquerda)))


for tabela in TABELAS_DIM:
    colunas = colunas_negocio('etl', tabela)
    mesmas_colunas = colunas == colunas_negocio('elt', tabela)
    registrar('6b equivalência', f'{tabela}: mesmos atributos', mesmas_colunas)
    if mesmas_colunas:
        campos = ', '.join(f'"{c}"' for c in colunas)
        consultas = [f'SELECT {campos}, COUNT(*) FROM {s}.{tabela} GROUP BY {campos}'
                     for s in ('etl', 'elt')]
        divergencias = diferencas_bidirecionais(*consultas)
        registrar('6b equivalência', f'{tabela}: conteúdo sem depender das SKs',
                  divergencias == 0, f'{divergencias} grupos divergentes')


def fato_canonica(esquema):
    campos = ['CAST(ROUND(f.valor_transacao * 100) AS INTEGER)']
    joins = []
    for i, (fk, tabela, pk) in enumerate(CHAVES):
        alias = f'd{i}'
        joins.append(f'JOIN {esquema}.{tabela} {alias} ON {alias}.{pk} = f.{fk}')
        campos.extend(f'{alias}."{c}"' for c in colunas_negocio(esquema, tabela))
    selecao = ', '.join(campos)
    return (f'SELECT {selecao}, COUNT(*) FROM {esquema}.{FATO} f '
            + ' '.join(joins) + f' GROUP BY {selecao}')


divergencias = diferencas_bidirecionais(fato_canonica('etl'), fato_canonica('elt'))
registrar('6b equivalência', 'transações: atributos, valores e multiplicidades iguais',
          divergencias == 0, f'{divergencias} grupos divergentes')

# Referência independente para o tratamento de sigilo, calculada nos CSVs.
marcadores = {'sigiloso', 'informações protegidas por sigilo'}
datas_especiais_csv = portadores_especiais_csv = 0
for caminho in ARQUIVOS_CSV:
    for r in ler_csv(caminho):
        sigilosa = (any(r[c].strip().casefold() in marcadores
                        for c in ('TRANSAÇÃO', 'NOME PORTADOR', 'NOME FAVORECIDO'))
                    or r['CNPJ OU CPF FAVORECIDO'].strip() == '-11')
        datas_especiais_csv += sigilosa or not r['DATA TRANSAÇÃO'].strip()
        portadores_especiais_csv += sigilosa or (not r['CPF PORTADOR'].strip() and not r['NOME PORTADOR'].strip())
for esquema in ('etl', 'elt'):
    sigilo = "(LOWER(t.descricao_original) IN ('sigiloso', 'informações protegidas por sigilo') OR LOWER(v.nome_favorecido) IN ('sigiloso', 'informações protegidas por sigilo') OR v.documento = '-11')"
    origem = f"{esquema}.{FATO} f JOIN {esquema}.DIM_TIPO_TRANSACAO t ON t.id_tipo_transacao=f.id_tipo_transacao JOIN {esquema}.DIM_FAVORECIDO v ON v.id_favorecido=f.id_favorecido"
    for coluna, esperado in [('id_data_transacao', datas_especiais_csv), ('id_portador', portadores_especiais_csv)]:
        obtido = escalar(f'SELECT COUNT(*) FROM {esquema}.{FATO} WHERE {coluna}=-1')
        registrar('6b sigilo', f'{esquema}: {coluna} especial = referência dos CSVs', obtido == esperado, f'{obtido} vs {esperado}')
    invalidos = escalar(f'SELECT COUNT(*) FROM {origem} WHERE {sigilo} AND (f.id_data_transacao != -1 OR f.id_portador != -1 OR f.id_mes_extrato = -1)')
    registrar('6b sigilo', f'{esquema}: marcadores de sigilo respeitam as chaves especiais', invalidos == 0)


[OK] DIM_TEMPO: mesmos atributos
[OK] DIM_TEMPO: conteúdo sem depender das SKs — 0 grupos divergentes
[OK] DIM_ORGAO: mesmos atributos
[OK] DIM_ORGAO: conteúdo sem depender das SKs — 0 grupos divergentes
[OK] DIM_PORTADOR: mesmos atributos
[OK] DIM_PORTADOR: conteúdo sem depender das SKs — 0 grupos divergentes
[OK] DIM_FAVORECIDO: mesmos atributos
[OK] DIM_FAVORECIDO: conteúdo sem depender das SKs — 0 grupos divergentes
[OK] DIM_TIPO_TRANSACAO: mesmos atributos
[OK] DIM_TIPO_TRANSACAO: conteúdo sem depender das SKs — 0 grupos divergentes
[OK] transações: atributos, valores e multiplicidades iguais — 0 grupos divergentes
[OK] etl: id_data_transacao especial = referência dos CSVs — 110172 vs 110172
[OK] etl: id_portador especial = referência dos CSVs — 110172 vs 110172
[OK] etl: marcadores de sigilo respeitam as chaves especiais
[OK] elt: id_data_transacao especial = referência dos CSVs — 110172 vs 110172
[OK] elt: id_portador especial = referência dos CSVs — 110172 vs 110172
[OK] elt: m

## 7. Características de execução

Além de tempo e memória, comparamos a **natureza** de cada abordagem com números que o notebook extrai sozinho: tamanho do banco, o que fica guardado nele, quantidade de código e dependências. Linhas de código contam apenas linhas não vazias e que não são comentários.

In [11]:
def linhas_de_codigo_notebook(caminho):
    notebook = json.loads(Path(caminho).read_text(encoding='utf-8'))
    total = 0
    for celula in notebook['cells']:
        if celula['cell_type'] == 'code':
            total += sum(1 for l in ''.join(celula['source']).splitlines() if l.strip() and not l.strip().startswith('#'))
    return total


def usa_pandas(caminho):
    notebook = json.loads(Path(caminho).read_text(encoding='utf-8'))
    return any(re.search(r'^\s*(import|from)\s+pandas', ''.join(c['source']), re.M)
               for c in notebook['cells'] if c['cell_type'] == 'code')


def tabelas(esquema):
    return [l[0] for l in conexao.execute(f"SELECT name FROM {esquema}.sqlite_master WHERE type='table' ORDER BY name")]


scripts_sql = sorted((RAIZ / 'scripts/sql/elt_transformations').glob('*.sql'))
linhas_sql = sum(1 for s in scripts_sql for l in s.read_text(encoding='utf-8').splitlines()
                 if l.strip() and not l.strip().startswith('--'))
tabelas_etl, tabelas_elt = tabelas('etl'), tabelas('elt')

caracteristicas = pd.DataFrame([
    ('Onde a transformação roda', 'Python / pandas (memória)', 'SQL dentro do SQLite'),
    ('Tempo total, mediana (s)', resumo_tempo.loc['ETL', 'mediana_s'], resumo_tempo.loc['ELT', 'mediana_s']),
    ('Tempo total, mínimo (s)', resumo_tempo.loc['ETL', 'minimo_s'], resumo_tempo.loc['ELT', 'minimo_s']),
    ('Pico de memória do kernel (MB)', round(resumo_tempo.loc['ETL', 'pico_memoria_mb']), round(resumo_tempo.loc['ELT', 'pico_memoria_mb'])),
    ('Tamanho do banco (MB)', round(DB_ETL.stat().st_size / 1e6, 2), round(DB_ELT.stat().st_size / 1e6, 2)),
    ('Tabelas no banco', len(tabelas_etl), len(tabelas_elt)),
    ('Guarda o dado bruto no banco', 'sim' if any(t.startswith('raw_') for t in tabelas_etl) else 'não',
     'sim' if any(t.startswith('raw_') for t in tabelas_elt) else 'não'),
    ('Linhas de código Python no notebook', linhas_de_codigo_notebook(RAIZ / 'notebooks/ETL.ipynb'),
     linhas_de_codigo_notebook(RAIZ / 'notebooks/ELT.ipynb')),
    ('Linhas de SQL de transformação', 0, linhas_sql),
    ('Scripts SQL de transformação', 0, len(scripts_sql)),
    ('Usa pandas', 'sim' if usa_pandas(RAIZ / 'notebooks/ETL.ipynb') else 'não',
     'sim' if usa_pandas(RAIZ / 'notebooks/ELT.ipynb') else 'não'),
], columns=['característica', 'ETL', 'ELT']).set_index('característica')
display(caracteristicas)

,ETL,ELT
característica,,
Onde a transformação roda,Python / pandas (memória),SQL dentro do SQLite
"Tempo total, mediana (s)",14.23,16.37
"Tempo total, mínimo (s)",13.66,15.73
Pico de memória do kernel (MB),678,80
Tamanho do banco (MB),54.91,302.96
Tabelas no banco,6,10
Guarda o dado bruto no banco,não,sim
Linhas de código Python no notebook,350,263
Linhas de SQL de transformação,0,235


## 8. Resumo final

Tabela com todas as verificações. Se houver `FALHA`, ela aparece em destaque, e a seção correspondente acima mostra os números para investigar.

In [12]:
relatorio = pd.DataFrame(resultados)
display(relatorio.groupby(['bloco', 'status']).size().unstack(fill_value=0))
falhas = relatorio[relatorio['status'] == 'FALHA']
if falhas.empty:
    print('Todas as verificações passaram.')
else:
    print('Verificações com FALHA:')
    display(falhas[['bloco', 'verificacao', 'detalhe']])
print(f"Resumo: {(relatorio['status'] == 'OK').sum()} OK | {len(falhas)} FALHA")

identificador_execucao = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
destino = RAIZ / 'data/processed' / f'benchmark_etl_elt_{"amostra" if MODO_AMOSTRA else "completo"}_{identificador_execucao}.json'
import hashlib
import platform
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=RAIZ, capture_output=True, text=True, check=True).stdout.strip()
entradas = {c.name: hashlib.sha256(c.read_bytes()).hexdigest() for c in ARQUIVOS_CSV}
destino.parent.mkdir(parents=True, exist_ok=True)
destino.write_text(json.dumps({
    'gerado_em': datetime.now().isoformat(timespec='seconds'),
    'modo': 'amostra' if MODO_AMOSTRA else 'completo',
    'repeticoes': REPETICOES,
    'commit': commit, 'sha256_entradas': entradas,
    'ambiente': {'python': sys.version, 'plataforma': platform.platform(), 'sqlite': sqlite3.sqlite_version, 'pandas': pd.__version__},
    'linhas_csv': linhas_csv, 'centavos_csv': centavos_csv,
    'tempos': medidas.round(3).to_dict(orient='records'),
    'resumo_tempo': resumo_tempo.reset_index().to_dict(orient='records'),
    'caracteristicas': caracteristicas.reset_index().astype(str).to_dict(orient='records'),
    'verificacoes': resultados,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Resultados salvos em', destino)

# Remove os arquivos temporários do benchmark.
for temporario in (RAIZ / 'notebooks').glob('_bench_*'):
    temporario.unlink()

status,OK
bloco,
4 contagem,9
5 soma,4
6 integridade,16
6b equivalência,11
6b sigilo,6


Todas as verificações passaram.
Resumo: 46 OK | 0 FALHA
Resultados salvos em /home/fagpaf/Projetos/projeto-integracao-dw/data/processed/benchmark_etl_elt_completo_20261117_193736_293958.json
